# 11 — Multilingual Embedding Experiments

## Goal
Experiment with a multilingual embedding model for cross-language semantic search.

**What happens using this code?** It embeds a manageable sample of Sinhala and English questions and finds semantically similar questions. These embeddings are for retrieval experiments, not supervised classification.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
df = pd.read_csv(ROOT / "datasets/processed/cleaned_paddy_qa.csv", keep_default_na=False, encoding="utf-8-sig")
# A sample keeps the experiment lightweight; remove n=... to embed the full table later.
sample = df.drop_duplicates(["question", "language"]).sample(n=min(500, df.shape[0]), random_state=42).reset_index(drop=True)
MODEL_NAME = "intfloat/multilingual-e5-base"
model = SentenceTransformer(MODEL_NAME)
passages = ["passage: " + q for q in sample["question"].tolist()]
embeddings = model.encode(passages, normalize_embeddings=True, batch_size=16, show_progress_bar=True)
print("Embedding matrix:", embeddings.shape)

# Example cross-language retrieval test: enter your own Sinhala or English question.
query = "වී පැළවල කොළ කහ පැහැයට හැරෙනවා"
q_vec = model.encode(["query: " + query], normalize_embeddings=True)
scores = cosine_similarity(q_vec, embeddings)[0]
top = np.argsort(scores)[::-1][:5]
results = sample.iloc[top][["question", "answer", "language", "category_en", "issue_id"]].copy()
results["similarity"] = scores[top]
display(results)

out = ROOT / "models/model_evaluation_results/embedding_experiment_sample.csv"
sample.to_csv(out, index=False, encoding="utf-8-sig")
print("Sample saved:", out)

**Important:** semantic similarity means wording or meaning may be related; it does not prove the retrieved answer is correct or safe. Treat this sample dataset as synthetic FAQ material until an agricultural expert checks it.